# Prediksi Gaji Menggunakan Linear Regression

**Tujuan:** Memprediksi gaji IT berdasarkan jabatan, tingkat pengalaman, jenis pekerjaan, ukuran perusahaan, lokasi, dan rasio kerja jarak jauh.

**Dataset:** salaries.csv berisi informasi gaji untuk profesional IT

**Metode:** Linear Regression

## 1. Import Library

In [ ]:
# Manipulasi data
import pandas as pd
import numpy as np

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Pengaturan tampilan
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
sns.set_palette('husl')

print('Library berhasil diimport!')

## 2. Memuat Dataset

In [ ]:
# Memuat dataset
df = pd.read_csv('salaries.csv')

# Menampilkan informasi dasar
print(f'Bentuk dataset: {df.shape}')
print(f'\nKolom: {list(df.columns)}')
print(f'\nBeberapa baris pertama:')
df.head()

In [ ]:
# Tipe data dan informasi
print('Informasi Dataset:')
df.info()

## 3. Analisis Data Eksploratif (EDA)

Menganalisis data untuk memahami pola dan mengidentifikasi fitur yang berkorelasi dengan gaji.

In [ ]:
# Memeriksa nilai yang hilang
print('Nilai yang Hilang:')
print(df.isnull().sum())
print(f'\nTotal nilai hilang: {df.isnull().sum().sum()}')

In [ ]:
# Ringkasan statistik
print('Ringkasan Statistik:')
df.describe()

In [ ]:
# Distribusi gaji
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(df['salary_in_usd'], bins=50, edgecolor='black')
plt.xlabel('Gaji (USD)')
plt.ylabel('Frekuensi')
plt.title('Distribusi Gaji')

plt.subplot(1, 2, 2)
plt.boxplot(df['salary_in_usd'])
plt.ylabel('Gaji (USD)')
plt.title('Boxplot Gaji')

plt.tight_layout()
plt.show()

print(f'Statistik Gaji:')
print(f'Rata-rata: ${df["salary_in_usd"].mean():,.2f}')
print(f'Median: ${df["salary_in_usd"].median():,.2f}')
print(f'Min: ${df["salary_in_usd"].min():,.2f}')
print(f'Maks: ${df["salary_in_usd"].max():,.2f}')

In [ ]:
# Tingkat pengalaman vs Gaji - menunjukkan korelasi
plt.figure(figsize=(10, 6))
df.boxplot(column='salary_in_usd', by='experience_level', figsize=(10, 6))
plt.xlabel('Tingkat Pengalaman')
plt.ylabel('Gaji (USD)')
plt.title('Gaji berdasarkan Tingkat Pengalaman')
plt.suptitle('')
plt.show()

print('\nRata-rata Gaji berdasarkan Tingkat Pengalaman:')
print(df.groupby('experience_level')['salary_in_usd'].mean().sort_values(ascending=False))

In [ ]:
# Ukuran perusahaan vs Gaji
plt.figure(figsize=(10, 6))
df.boxplot(column='salary_in_usd', by='company_size', figsize=(10, 6))
plt.xlabel('Ukuran Perusahaan')
plt.ylabel('Gaji (USD)')
plt.title('Gaji berdasarkan Ukuran Perusahaan')
plt.suptitle('')
plt.show()

print('\nRata-rata Gaji berdasarkan Ukuran Perusahaan:')
print(df.groupby('company_size')['salary_in_usd'].mean().sort_values(ascending=False))

In [ ]:
# Rasio remote vs Gaji
plt.figure(figsize=(10, 6))
df.boxplot(column='salary_in_usd', by='remote_ratio', figsize=(10, 6))
plt.xlabel('Rasio Remote (%)')
plt.ylabel('Gaji (USD)')
plt.title('Gaji berdasarkan Rasio Kerja Jarak Jauh')
plt.suptitle('')
plt.show()

print('\nRata-rata Gaji berdasarkan Rasio Remote:')
print(df.groupby('remote_ratio')['salary_in_usd'].mean().sort_values(ascending=False))

In [ ]:
# Analisis variabel kategorikal
print('Distribusi Tipe Pekerjaan:')
print(df['employment_type'].value_counts())
print('\n10 Jabatan Teratas:')
print(df['job_title'].value_counts().head(10))
print(f'\nTotal jabatan unik: {df["job_title"].nunique()}')

## 4. Preprocessing Data

Mengelompokkan jabatan yang mirip ke dalam kategori untuk mengurangi dimensi dan meningkatkan performa model.

In [ ]:
# Fungsi untuk mengelompokkan jabatan ke dalam kategori
def group_job_title(title):
    title_lower = title.lower()
    if 'data scientist' in title_lower or 'data science' in title_lower:
        return 'Data Scientist'
    elif 'engineer' in title_lower:
        return 'Engineer'
    elif 'manager' in title_lower or 'director' in title_lower or 'head' in title_lower or 'lead' in title_lower:
        return 'Manager/Leadership'
    elif 'analyst' in title_lower or 'analytics' in title_lower:
        return 'Analyst'
    elif 'ml' in title_lower or 'machine learning' in title_lower or 'ai' in title_lower or 'artificial intelligence' in title_lower:
        return 'ML/AI Specialist'
    else:
        return 'Lainnya'

# Menerapkan pengelompokan
df['job_category'] = df['job_title'].apply(group_job_title)

print('Distribusi Kategori Jabatan:')
print(df['job_category'].value_counts())
print('\nRata-rata Gaji berdasarkan Kategori Jabatan:')
print(df.groupby('job_category')['salary_in_usd'].mean().sort_values(ascending=False))

## 5. Encoding Fitur

Mengonversi variabel kategorikal ke format numerik untuk model linear regression.

In [ ]:
# Membuat salinan untuk preprocessing
df_processed = df.copy()

# Label encode experience_level (ordinal: EN < MI < SE < EX)
experience_mapping = {'EN': 0, 'MI': 1, 'SE': 2, 'EX': 3}
df_processed['experience_level_encoded'] = df_processed['experience_level'].map(experience_mapping)

print('Encoding Tingkat Pengalaman:')
print(df_processed[['experience_level', 'experience_level_encoded']].drop_duplicates().sort_values('experience_level_encoded'))

In [ ]:
# One-hot encode variabel kategorikal
categorical_features = ['employment_type', 'job_category', 'company_size']

df_encoded = pd.get_dummies(df_processed, columns=categorical_features, drop_first=True)

print(f'Bentuk awal: {df_processed.shape}')
print(f'Bentuk setelah encoding: {df_encoded.shape}')
print(f'\nKolom baru setelah encoding: {df_encoded.shape[1] - df_processed.shape[1]} ditambahkan')

## 6. Pelatihan Model

Membagi data dan melatih model linear regression untuk memprediksi gaji.

In [ ]:
# Memilih fitur untuk model
feature_columns = ['work_year', 'experience_level_encoded', 'remote_ratio'] + \
                  [col for col in df_encoded.columns if col.startswith(('employment_type_', 'job_category_', 'company_size_'))]

X = df_encoded[feature_columns]
y = df_encoded['salary_in_usd']

print(f'Fitur dipilih: {len(feature_columns)}')
print(f'Nama fitur: {feature_columns[:10]}...')  # Menampilkan 10 pertama
print(f'\nBentuk X: {X.shape}')
print(f'Bentuk y: {y.shape}')

In [ ]:
# Membagi data menjadi training dan testing (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Ukuran training set: {X_train.shape[0]} sampel')
print(f'Ukuran testing set: {X_test.shape[0]} sampel')
print(f'\nTraining set: {X_train.shape[0]/len(X)*100:.1f}%')
print(f'Testing set: {X_test.shape[0]/len(X)*100:.1f}%')

In [ ]:
# Membuat dan melatih model Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

print('Model Linear Regression Berhasil Dilatih!')
print(f'\nIntercept Model: ${model.intercept_:,.2f}')
print(f'\n10 Koefisien Fitur Teratas:')
coef_df = pd.DataFrame({'Fitur': feature_columns, 'Koefisien': model.coef_})
coef_df = coef_df.sort_values('Koefisien', ascending=False)
print(coef_df.head(10))

## 7. Evaluasi Model

Mengevaluasi performa model menggunakan berbagai metrik dan visualisasi.

In [ ]:
# Membuat prediksi pada testing set
y_pred = model.predict(X_test)

# Menghitung metrik evaluasi
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print('Metrik Performa Model:')
print(f'R² Score: {r2:.4f}')
print(f'Mean Absolute Error (MAE): ${mae:,.2f}')
print(f'Mean Squared Error (MSE): ${mse:,.2f}')
print(f'Root Mean Squared Error (RMSE): ${rmse:,.2f}')
print(f'\nInterpretasi: Model menjelaskan {r2*100:.2f}% varians dalam gaji.')

In [ ]:
# Scatter plot Aktual vs Prediksi
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Gaji Aktual (USD)')
plt.ylabel('Gaji Prediksi (USD)')
plt.title('Aktual vs Prediksi Gaji')
plt.tight_layout()
plt.show()

In [ ]:
# Plot residual
residuals = y_test - y_pred

plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Gaji Prediksi (USD)')
plt.ylabel('Residual (USD)')
plt.title('Plot Residual')
plt.tight_layout()
plt.show()

## 8. Membuat Prediksi

Menggunakan model yang telah dilatih untuk memprediksi gaji data baru.

In [ ]:
# Contoh: Memprediksi gaji untuk individu baru
# Senior Engineer, Full-time, Perusahaan Medium, 100% remote, Tahun 2025

# Membuat data sampel sesuai dengan encoding kita
sample_data = pd.DataFrame({
    'work_year': [2025],
    'experience_level_encoded': [2],  # SE (Senior)
    'remote_ratio': [100]
})

# Menambahkan fitur encoding (set semua ke 0, lalu set yang diinginkan ke 1)
for col in feature_columns:
    if col not in sample_data.columns:
        sample_data[col] = 0

# Mengatur fitur kategorikal spesifik
if 'employment_type_FT' in feature_columns:
    sample_data['employment_type_FT'] = 1
if 'job_category_Engineer' in feature_columns:
    sample_data['job_category_Engineer'] = 1
if 'company_size_M' in feature_columns:
    sample_data['company_size_M'] = 1

# Menata ulang kolom agar sesuai dengan data training
sample_data = sample_data[feature_columns]

# Membuat prediksi
predicted_salary = model.predict(sample_data)[0]

print('Contoh Prediksi:')
print('Profil: Senior Engineer, Full-time, Perusahaan Medium, 100% remote, Tahun 2025')
print(f'Gaji Prediksi: ${predicted_salary:,.2f} USD')

## 9. Kesimpulan

**Temuan Utama:**
- Tingkat pengalaman menunjukkan korelasi kuat dengan gaji (pengalaman lebih tinggi = gaji lebih tinggi)
- Kategori jabatan (Manager/Leadership, Engineer, Data Scientist, dll.) secara signifikan memengaruhi gaji
- Ukuran perusahaan dan rasio kerja jarak jauh juga memengaruhi kompensasi
- Model linear regression berhasil memprediksi gaji IT berdasarkan faktor-faktor ini

**Performa Model:**
- Skor R² menunjukkan seberapa baik model menjelaskan varians gaji
- RMSE menunjukkan rata-rata kesalahan prediksi dalam USD

**Langkah Selanjutnya:**
- Pertimbangkan feature engineering (interaksi, fitur polinomial)
- Coba model regresi lain (Ridge, Lasso, Random Forest)
- Analisis pola gaji spesifik berdasarkan lokasi